<a href="https://colab.research.google.com/github/google-ai-edge/litert-samples/blob/main/compiled_model_api/colab/On_Device_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2026 The AI Edge Authors.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# On Device RAG using Lite-RT Gemma-4 and Qdrant Edge

### Overview

Build a fully on-device RAG pipeline that runs entirely without any cloud API calls. It combines three local components: a Qdrant edge vector store for retrieval, a Google Embedding Gemma model for semantic search, and Gemma 4 via LiteRT-LM for answer generation with all running locally on CPU. 

- Vector store setup: Uses qdrant-edge-py to create a local EdgeShard with cosine similarity, then index 10 sample documents encoded ``via google/embeddinggemma-300m``
- Semantic retrieval: Encodes the user query with the same embedding model and runs a nearest-neighbor search on the shard to pull the top-2 most relevant document chunks as context.
- On-device LLM inference: Feeds the retrieved context into Gemma-4-E2B running locally via litert_lm.Engine on CPU backend, with a strict RAG system prompt instructing the model to only answer from the provided context.

## Installation

Install the Python dependencies and the Vulkan library needed by LiteRT-LM's backend. Since we will be running on CPU no need to switch to the hardware accelerator. 

In [1]:
!uv pip install litert-lm-api qdrant-edge-py sentence-transformers

Using Python 3.12.13 environment at: /usr
Resolved 41 packages in 681ms                                        
Prepared 2 packages in 889ms                                             
Installed 2 packages in 1ms1                                
 + litert-lm-api==0.13.1
 + qdrant-edge-py==0.7.2


In [2]:
!apt-get install -y libvulkan1

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  mesa-vulkan-drivers
The following NEW packages will be installed:
  libvulkan1 mesa-vulkan-drivers
0 upgraded, 2 newly installed, 0 to remove and 3 not upgraded.
Need to get 10.9 MB of archives.
After this operation, 51.3 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libvulkan1 amd64 1.3.204.1-2 [128 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 mesa-vulkan-drivers amd64 23.2.1-1ubuntu3.1~22.04.3 [10.7 MB]
Fetched 10.9 MB in 0s (58.3 MB/s)             
Selecting previously unselected package libvulkan1:amd64.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../libvulkan1_1.3.204.1-2_amd64.deb ...
Unpacking libvulkan1:amd64 (1.3.204.1-2) ...
Selecting previously unselected package mesa-vulkan-drivers:amd64.
Preparing to 

## Initial Setup

To use [google/embeddinggemma-300m](https://huggingface.co/google/embeddinggemma-300m), you first need to accept the model terms on its Hugging Face page and then generate a token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and paste it below.


In [3]:
import os
import shutil

from sentence_transformers import SentenceTransformer
from qdrant_edge import (
    Distance,EdgeConfig,EdgeShard,EdgeVectorParams,
    Point,Query,QueryRequest,UpdateOperation,
)

import litert_lm

In [ ]:
os.environ["HF_TOKEN"] = "<replace-with-hf-token>"

In [6]:
MODEL_DIM = 768
MODEL_NAME = "google/embeddinggemma-300m"

path = "./tmp/qdrant_edge_eg"
shutil.rmtree(path, ignore_errors=True)
os.makedirs(path)

In [ ]:
model = SentenceTransformer(MODEL_NAME)
shard = EdgeShard.create(
    path,
    EdgeConfig(vectors=EdgeVectorParams(size=MODEL_DIM, distance=Distance.Cosine)),
)

## Index your document

Encode all documents into 768-dim vectors using the embedding model, then upsert them into the Qdrant edge shard as points with their text as payload. shard.optimize() finalizes the index for faster query performance.

In [8]:
documents = [
    (1, "The bakery opens just before dawn, when the streets are still quiet and the first customers begin walking in for fresh bread. The smell of warm sourdough fills the small shop as the bakers arrange loaves, croissants, and pastries behind the counter. Many people stop there every morning before work because the bakery has become a familiar part of the neighborhood."),
    (2, "A golden retriever ran happily across the park after its owner threw a bright tennis ball into the grass. Children nearby stopped to watch as the dog jumped, rolled, and carried the ball back with excitement. The park was full of people enjoying the sunny afternoon, but the playful dog easily became the center of attention."),
    (3, "Python decorators are a powerful feature used to wrap functions and extend their behavior without changing the original function code. Developers often use decorators for logging, caching, authentication, and performance tracking. They make the code cleaner by separating repeated logic from the main business logic."),
    (4, "Heavy rain continued through the night and caused several streets to flood by early morning. Commuters had to wait longer than usual because buses were delayed and traffic moved slowly across the city. Many people arrived late to work, while others decided to stay home until the weather improved."),
    (5, "She booked a window seat on the overnight train to Vienna because she wanted to watch the quiet towns pass by during the journey. Her dog rested beside her in a small travel carrier, occasionally waking up whenever the train stopped at a station. The trip felt peaceful, and she enjoyed the comfort of traveling slowly through the night."),
    (6, "The old library stood at the corner of the main road, surrounded by tall trees and a quiet garden. Students often came there after school to study, read novels, or prepare for exams in the large reading hall. Even though the building was old, it remained one of the most loved places in the town."),
    (7, "The startup team worked late into the evening to finish the first version of their product before the investor demo. Everyone had a different responsibility, from fixing bugs to improving the user interface and preparing the final presentation. By midnight, they were tired but excited because the product finally started to feel real."),
    (8, "The mountain village was covered in mist when the travelers arrived early in the morning. Small houses with wooden balconies stood along narrow paths, and smoke rose slowly from kitchen chimneys. The place felt calm and untouched, making it perfect for people who wanted to escape the noise of the city."),
    (9, "The school science fair attracted students, parents, and teachers from across the district. Each team presented a different project, including simple robots, water filtration models, and solar powered devices. The event encouraged students to explain their ideas clearly and learn from what others had built."),
    (10, "The restaurant near the beach became popular because of its simple food, friendly staff, and beautiful sunset view. Tourists usually visited in the evening, while locals preferred coming during quiet afternoons. The owner personally greeted regular customers and made sure every table felt comfortable and welcome."),
]

In [9]:
texts = [text for _, text in documents]
embeddings = model.encode_document(texts)

In [11]:
shard.update(UpdateOperation.upsert_points([
    Point(
        point_id,
        embeddings[i].tolist(),
        {"text": text},
    )
    for i, (point_id, text) in enumerate(documents)
]))
shard.optimize()

False

### Test your retrieval

Ask a user query and retrieve the top_k = 2 relevant documents from the Qdrant-Edge knowledge store. 

In [10]:
user_query = "Which animal chased the tennis ball?"

In [12]:
query_vector = model.encode_query(user_query).tolist()

In [13]:
results = shard.query(QueryRequest(
    query=Query.Nearest(query_vector),
    limit=2,
    with_payload=True,
))

In [14]:
context = ""
for point in results:
  context+= point.payload['text']

print(context)

A golden retriever ran happily across the park after its owner threw a bright tennis ball into the grass. Children nearby stopped to watch as the dog jumped, rolled, and carried the ball back with excitement. The park was full of people enjoying the sunny afternoon, but the playful dog easily became the center of attention.She booked a window seat on the overnight train to Vienna because she wanted to watch the quiet towns pass by during the journey. Her dog rested beside her in a small travel carrier, occasionally waking up whenever the train stopped at a station. The trip felt peaceful, and she enjoyed the comfort of traveling slowly through the night.


## Define System Prompt

Set up the system message that instructs Gemma to act as a RAG assistant answering strictly from the retrieved context and keeping responses concise. This is passed into the conversation at engine initialization.

In [16]:
messages = [
    litert_lm.Message.system(
        "You are a helpful RAG assistant. "
        "Answer the user question using only the provided context. "
        "If the answer is not in the context, say: I don't know based on the provided context. "
        "Keep the answer short."
    )
]

## On-Device LLM Inference using Lite-RT LM

- Download only the base gemma-4-E2B-it.litertlm file (~2.6GB) from HuggingFace instead of the full repo. 
- Initialize the LiteRT-LM engine on CPU backend with the system prompt. 
- Answer Generation: The conversation sends the retrieved context and user query to Gemma 4, which generates an answer strictly grounded in the provided context. 

In [17]:
!hf download litert-community/gemma-4-E2B-it-litert-lm gemma-4-E2B-it.litertlm --local-dir ~/gemma4

Hint: A new version of huggingface_hub (1.19.0) is available! You are using version 1.18.0.
To update, run: hf update
gemma-4-E2B-it.litertlm: 100% 2.59G/2.59G [00:21<00:00, 122MB/s] 
✓ Downloaded
  path: /root/gemma4/gemma-4-E2B-it.litertlm


In [18]:
!ls /root/gemma4

gemma-4-E2B-it.litertlm


In [19]:
GEMMA4_LITELLM = "/root/gemma4/gemma-4-E2B-it.litertlm"

In [20]:
with litert_lm.Engine(
    GEMMA4_LITELLM,
    backend=litert_lm.Backend.CPU(),
) as engine:
    with engine.create_conversation(messages=messages) as conversation:
        response = conversation.send_message(f"Context:\n{context}\n\nQuestion:\n{user_query}")
        print(response["content"][0]["text"])

A golden retriever chased the tennis ball.
